In [49]:
import numpy as np
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from services.files import get_files
from services.distortion import hard_clip
from services.distortion import soft_clip
from services.distortion import bitcrush
from services.distortion import wavefold
from services.distortion import tremolo
from services.distortion import reverb
from sklearn.model_selection import train_test_split
import os
import librosa
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.linear_model import LogisticRegression
import time
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [50]:

torch.set_default_device('cpu')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [51]:
files = get_files()
print(len(files))
print(files[:5])

2913
['TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#1-ff-N-T30d.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#1-mf-N-N.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#1-pp-N-N.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#2-ff-N-T19u.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#2-mf-N-T29u.wav']


In [52]:
AudioData = []
TargetSeconds = 3
TargetSamples = TargetSeconds * 44100 

for file in files:
    audio, sr = librosa.load(file, sr=44100)
    audio = audio/ np.max(np.abs(audio))
    audio_fixed = librosa.util.fix_length(audio, size=TargetSamples)
    AudioData.append(audio_fixed)


AudioData = np.array(AudioData)

print(AudioData.shape)

print(AudioData,AudioData[0].shape, AudioData[1].shape)

(2913, 132300)
[[ 0.0000000e+00  0.0000000e+00  1.4452955e-04 ...  0.0000000e+00
   0.0000000e+00  0.0000000e+00]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ... -2.1752988e-01
  -2.2788845e-01 -2.3585658e-01]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ... -1.0321101e-01
  -1.0550459e-01 -1.0550459e-01]
 ...
 [ 0.0000000e+00 -1.6079756e-04 -1.6079756e-04 ... -7.2809136e-01
  -7.1410197e-01 -5.9093100e-01]
 [ 0.0000000e+00  0.0000000e+00  2.3849272e-04 ... -3.5606965e-01
  -2.5017887e-01 -7.0593849e-02]
 [ 1.1376564e-03  0.0000000e+00  0.0000000e+00 ... -5.9726965e-01
  -5.6882823e-01 -5.3924912e-01]] (132300,) (132300,)


In [53]:
X_train_clean, X_test_clean = train_test_split(
    AudioData, test_size=0.2, random_state=42, shuffle=True
)

In [54]:
X_train = X_train.reshape(len(X_train), -1)
X_test  = X_test.reshape(len(X_test), -1)

In [55]:
def apply_random_distortion(x):
    effects = [
        lambda x: hard_clip(x, 0.5),
        lambda x: soft_clip(x, 3),
        lambda x: bitcrush(x),
        lambda x: wavefold(x),
        lambda x: tremolo(x),
        lambda x: reverb(x)
    ]
    
    effect = np.random.choice(effects)
    distorted = effect(x)

    # Normalize after distortion
    distorted = distorted / (np.max(np.abs(distorted)) + 1e-8)

    return distorted

In [56]:
def build_binary_dataset(clean_data):
    X = []
    y = []

    for signal in clean_data:
        # Clean sample
        X.append(signal)
        y.append(0)

        # Distorted sample
        distorted = apply_random_distortion(signal)
        X.append(distorted)
        y.append(1)

    X = np.array(X)
    y = np.array(y)

    # Shuffle
    perm = np.random.permutation(len(X))
    return X[perm], y[perm]

In [57]:
X_train, y_train = build_binary_dataset(X_train_clean)
X_test, y_test = build_binary_dataset(X_test_clean)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(4660, 132300) (4660,)
(1166, 132300) (1166,)


In [64]:
def extract_features(X):
    features = []

    for signal in X:
        # ===== Time domain =====
        mean = np.mean(signal)
        std = np.std(signal)
        energy = np.mean(np.abs(signal))
        rms = np.sqrt(np.mean(signal**2))
        peak = np.max(np.abs(signal))

        # Crest factor (VERY important for distortion)
        crest = peak / (rms + 1e-8)

        # Zero crossing rate
        zcr = np.mean(np.abs(np.diff(np.sign(signal))))

        # ===== Frequency domain =====
        fft = np.fft.rfft(signal)
        mag = np.abs(fft)

        fft_mean = np.mean(mag)
        fft_std = np.std(mag)

        # Spectral centroid (VERY useful)
        freqs = np.fft.rfftfreq(len(signal), d=1/44100)
        centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-8)

        # Spectral bandwidth
        bandwidth = np.sqrt(np.sum(((freqs - centroid)**2) * mag) / (np.sum(mag) + 1e-8))

        features.append([
            mean, std, energy, rms, peak,
            crest, zcr,
            fft_mean, fft_std,
            centroid, bandwidth
        ])

    return np.array(features)

In [65]:
X_train_feat = extract_features(X_train)
X_test_feat  = extract_features(X_test)

In [66]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat  = scaler.transform(X_test_feat)

In [67]:
X_train = X_train.reshape(len(X_train), -1)
X_test  = X_test.reshape(len(X_test), -1)



In [68]:
model = SGDClassifier(loss="log_loss", max_iter=1, warm_start=True)
epochs = 100

for epoch in range(epochs):
    model.fit(X_train_feat, y_train)
    
    train_acc = model.score(X_train_feat, y_train)
    test_acc = model.score(X_test_feat, y_test)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

Epoch 1/100 | Train Acc: 0.7343 | Test Acc: 0.7350
Epoch 2/100 | Train Acc: 0.6678 | Test Acc: 0.6544
Epoch 3/100 | Train Acc: 0.7159 | Test Acc: 0.7213
Epoch 4/100 | Train Acc: 0.7414 | Test Acc: 0.7461
Epoch 5/100 | Train Acc: 0.7090 | Test Acc: 0.7144
Epoch 6/100 | Train Acc: 0.6592 | Test Acc: 0.6810
Epoch 7/100 | Train Acc: 0.7348 | Test Acc: 0.7264
Epoch 8/100 | Train Acc: 0.7148 | Test Acc: 0.7101
Epoch 9/100 | Train Acc: 0.7122 | Test Acc: 0.7221
Epoch 10/100 | Train Acc: 0.6644 | Test Acc: 0.6741
Epoch 11/100 | Train Acc: 0.7197 | Test Acc: 0.7221
Epoch 12/100 | Train Acc: 0.6764 | Test Acc: 0.6835
Epoch 13/100 | Train Acc: 0.6131 | Test Acc: 0.6046
Epoch 14/100 | Train Acc: 0.7554 | Test Acc: 0.7461
Epoch 15/100 | Train Acc: 0.6931 | Test Acc: 0.6810
Epoch 16/100 | Train Acc: 0.7182 | Test Acc: 0.7153
Epoch 17/100 | Train Acc: 0.5861 | Test Acc: 0.5789
Epoch 18/100 | Train Acc: 0.6043 | Test Acc: 0.5755
Epoch 19/100 | Train Acc: 0.7333 | Test Acc: 0.7307
Epoch 20/100 | Train 

c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Dist

In [69]:
y_pred = model.predict(X_test_feat)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 0.7152658662092625

Report:
               precision    recall  f1-score   support

           0       0.66      0.87      0.75       583
           1       0.81      0.56      0.66       583

    accuracy                           0.72      1166
   macro avg       0.74      0.72      0.71      1166
weighted avg       0.74      0.72      0.71      1166

